# Lab 13 - Similarity Based Learning - K Nearest Neighbor
## Building a KNN Classifier from Scratch

## 1 Introduction & 2 Getting Started

In [1]:
from scipy.spatial.distance import euclidean as euc
import numpy as np
np.random.seed(0)

In [16]:
class KNN:
    def __init__(self):
        pass

    def fit(self, X_train, y_train):
        pass

    def predict(self, X_test, k=3):
        pass

## 2.1 Completing the `fit` Method

In [3]:
def fit(self, X_train, y_train):
    self.X_train = X_train
    self.y_train = y_train

# This line updates the knn.fit method to point to the function we've just written
KNN.fit = fit

### 2.1.1 Helper Functions

In [4]:
def _get_distances(self, x):
    distances = []
    for index, point in enumerate(self.X_train):
        dist_to_point = euc(x, point)
        distances.append((index, dist_to_point))
    return distances

# This line attaches the function we just created as a method to our KNN class.
KNN._get_distances = _get_distances

In [5]:
def _get_k_nearest(self, dists, k):
    sorted_dists = sorted(dists, key=lambda x: x[1])
    return sorted_dists[:k]

# This line attaches the function we just created as a method to our KNN class.
KNN._get_k_nearest = _get_k_nearest

In [6]:
def _get_label_prediction(self, k_nearest):
    labels = [self.y_train[index] for index, dist in k_nearest]
    counts = np.bincount(labels)
    return np.argmax(counts)

# This line attaches the function we just created as a method to our KNN class.
KNN._get_label_prediction = _get_label_prediction

## 2.2 Completing the `predict` Method

In [7]:
def predict(self, X_test, k=3):
    preds = []
    for x in X_test:
        dists = self._get_distances(x)
        k_nearest = self._get_k_nearest(dists, k)
        pred = self._get_label_prediction(k_nearest)
        preds.append(pred)
    return preds

KNN.predict = predict

## 2.3 Testing Our KNN Classifier

In [8]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
data = iris.data
target = iris.target

In [9]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.25, random_state=0)

In [10]:
knn = KNN()
knn.fit(X_train, y_train)

In [11]:
preds = knn.predict(X_test)

In [12]:
print("Testing Accuracy: {}".format(accuracy_score(y_test, preds)))

Testing Accuracy: 0.9736842105263158


## 2.4 Summary

The from-scratch KNN classifier achieves high accuracy on the Iris dataset, comparable to
scikit-learn's built-in `KNeighborsClassifier`. The implementation works by, for each test point,
calculating the Euclidean distance to every training point, selecting the k closest points, and
predicting the most common label among those neighbors (majority vote).

# Weighted KNN

Now let's implement a weighted version of KNN, where closer neighbors have a larger influence on the prediction than farther ones.

In [13]:
class WeightedKNN:
    def __init__(self):
        pass

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def _get_distances(self, x):
        distances = []
        for index, point in enumerate(self.X_train):
            dist_to_point = euc(x, point)
            distances.append((index, dist_to_point))
        return distances

    def _get_k_nearest(self, dists, k):
        sorted_dists = sorted(dists, key=lambda x: x[1])
        return sorted_dists[:k]

    def _get_weighted_label_prediction(self, k_nearest):
        # Weight each neighbor's vote by the inverse of its distance
        # (adding a small epsilon to avoid division by zero)
        epsilon = 1e-5
        class_weights = {}
        for index, dist in k_nearest:
            label = self.y_train[index]
            weight = 1 / (dist + epsilon)
            class_weights[label] = class_weights.get(label, 0) + weight
        # Return the class with the highest total weight
        return max(class_weights, key=class_weights.get)

    def predict(self, X_test, k=3):
        preds = []
        for x in X_test:
            dists = self._get_distances(x)
            k_nearest = self._get_k_nearest(dists, k)
            pred = self._get_weighted_label_prediction(k_nearest)
            preds.append(pred)
        return preds

In [14]:
wknn = WeightedKNN()
wknn.fit(X_train, y_train)
weighted_preds = wknn.predict(X_test)

print("Weighted KNN Testing Accuracy: {}".format(accuracy_score(y_test, weighted_preds)))

Weighted KNN Testing Accuracy: 0.9736842105263158


## Comparing Standard KNN vs Weighted KNN

In [15]:
for k in [1, 3, 5, 7, 9]:
    knn_preds = knn.predict(X_test, k=k)
    wknn_preds = wknn.predict(X_test, k=k)
    knn_acc = accuracy_score(y_test, knn_preds)
    wknn_acc = accuracy_score(y_test, wknn_preds)
    print(f"k={k}: Standard KNN accuracy = {knn_acc:.4f}, Weighted KNN accuracy = {wknn_acc:.4f}")

k=1: Standard KNN accuracy = 0.9737, Weighted KNN accuracy = 0.9737
k=3: Standard KNN accuracy = 0.9737, Weighted KNN accuracy = 0.9737
k=5: Standard KNN accuracy = 0.9737, Weighted KNN accuracy = 0.9737
k=7: Standard KNN accuracy = 0.9737, Weighted KNN accuracy = 0.9737
k=9: Standard KNN accuracy = 0.9737, Weighted KNN accuracy = 0.9737


## Comparison & Discussion

On the Iris dataset, both standard (unweighted) KNN and distance-weighted KNN achieve very high
accuracy (often around 95-100%) because the classes are largely well separated and the dataset is
small and clean. The differences between the two strategies become more noticeable when:

- **k is large**: with unweighted KNN, far-away points get the same vote as very close points,
  which can dilute the prediction. Weighted KNN reduces this effect since far neighbors contribute
  less.
- **The data is noisy or classes overlap**: weighted KNN tends to be more robust because it gives
  more importance to the closest (most relevant) neighbors.
- **k is small (e.g., k=1)**: both approaches behave identically, since with a single neighbor
  there's nothing to weight against.

Overall, weighted KNN is generally a safer default since it degrades gracefully as k increases,
whereas standard KNN's accuracy can drop more sharply for large k on noisier datasets.
